# 线性模型 Lasso / Ridge — Walk-Forward 正则化回归

本 notebook 用连续收益标签进行回归建模，并将预测值作为股票排序分数。

## Step 0 — 参数配置

| 参数 | 默认值 | 用途 |
|------|--------|------|
| `FOLD_DIR` | `00_fold切分/.../output/20260526` | fold CSV 目录 |
| `FOLD_ID` | `1` | 当前 fold 编号 |
| `LABEL_COL` | `FWD_RET_10D` | 标签列名 |
| `WINSORIZE_MAD` | `3.0` | MAD 倍数 |
| `IC_THRESHOLD` | `0.02` | IC 初筛阈值 |
| `T_STAT_THRESHOLD` | `2.0` | t 统计量阈值 |
| `CORR_THRESHOLD` | `0.7` | 共线性阈值 |
| `EWM_HALFLIFE` | `20` | β 平滑半衰期 |
| `MODEL_TYPE` | `lasso` | 模型类型：`lasso` 或 `ridge` |
| `ALPHA` | `0.001` | 正则化强度，越大惩罚越重 |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.linear_model import Lasso, Ridge
from typing import List, Dict, Tuple, Optional
import datetime as _dt
import warnings
warnings.filterwarnings("ignore")

# ===== 参数区 =====
FOLD_DIR         = Path("D:/quant/ML_teamwork/00_fold切分/01_train&test/output/20260526")
FOLD_ID          = 1
LABEL_COL        = "FWD_RET_10D"
WINSORIZE_MAD    = 3.0
IC_THRESHOLD     = 0.02
T_STAT_THRESHOLD = 2.0
CORR_THRESHOLD   = 0.7
EWM_HALFLIFE     = 20
OUTPUT_DIR       = Path("D:/quant/ML_teamwork/01_线性模型/02_线性模型lasso_ridge/output")

# ===== 新增参数 =====
MODEL_TYPE = "lasso"   # "lasso" 或 "ridge"
ALPHA      = 0.001     # 正则化强度

print("参数配置完成")
print(f"  Fold 目录: {FOLD_DIR}")
print(f"  Fold ID: {FOLD_ID}")
print(f"  标签列: {LABEL_COL}")
print(f"  Winsorize MAD: {WINSORIZE_MAD}")
print(f"  IC 阈值: {IC_THRESHOLD}  t 阈值: {T_STAT_THRESHOLD}")
print(f"  共线性阈值: {CORR_THRESHOLD}")
print(f"  EWM halflife: {EWM_HALFLIFE}")
print(f"  模型类型: {MODEL_TYPE.upper()}")
print(f"  Alpha (正则化强度): {ALPHA}")

## Step 1 — 函数定义

共 15 个函数，在 baseline 基础上新增 `daily_regularized_beta`，其余 14 个与 baseline 完全一致。

| 函数 | 用途 | 变化 |
|------|------|------|
| `load_fold` | 加载 fold CSV | 不变 |
| `identify_columns` | 识别特征列和标签列 | 不变 |
| `cross_section_winsorize` | 逐日截面 MAD 去极值 | 不变 |
| `cross_section_demean` | 逐日截面 demean | 不变 |
| `cross_section_zscore` | 逐日截面 z-score | 不变 |
| `compute_ic_panel` | 逐日 Spearman IC 面板 | 不变 |
| `summarize_ic` | IC 汇总统计 | 不变 |
| `filter_by_ic` | 阈值过滤因子 | 不变 |
| `spearman_union_find` | Spearman 相关矩阵 + 并查集 | 不变 |
| `select_representatives` | 每组取 \|IC\| 最大代表 | 不变 |
| **`daily_regularized_beta`** | **逐日 Lasso/Ridge 拟合** | **新增** |
| `smooth_beta_ewm` | β 时间序列 EWM 平滑 | 不变 |
| `predict_walkforward` | Walk-forward 预测 | 不变 |
| `evaluate_ic` | 日度 IC + 汇总评估 | 不变 |

In [ ]:
# ===== 数据加载 & 列识别（与 baseline 完全相同）=====

def load_fold(fold_dir: Path, fold_id: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """加载指定 fold 的训练/测试 CSV。

    输入:
        fold_dir: CSV 文件所在目录 (Path)
        fold_id: fold 编号 (int)

    输出:
        (train_df, test_df): 均已按 trade_date + ts_code 排序
    """
    tr = pd.read_csv(fold_dir / f"fold_{fold_id}_train.csv", parse_dates=["trade_date"])
    te = pd.read_csv(fold_dir / f"fold_{fold_id}_test.csv", parse_dates=["trade_date"])
    tr = tr.sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
    te = te.sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
    return tr, te


def identify_columns(df: pd.DataFrame) -> Tuple[List[str], str]:
    """识别特征列和标签列。

    输入:
        df: 因子表 DataFrame

    输出:
        (feature_cols, label_col): ALPHA_* 列名列表, 标签列名
    """
    feat = [c for c in df.columns if c.startswith("ALPHA_")]
    labels = [c for c in df.columns if c.startswith("FWD_RET_")]
    assert len(labels) >= 1, f"未找到 FWD_RET_* 标签列, cols={df.columns.tolist()}"
    label = labels[0]
    assert all(not c.startswith("FWD_RET_") for c in feat), "特征列含未来收益字段!"
    return feat, label


print("[OK] 数据函数定义完成")

In [ ]:
# ===== 截面预处理（与 baseline 完全相同）=====

def cross_section_winsorize(df: pd.DataFrame, cols: List[str], n_mad: float = 3.0) -> pd.DataFrame:
    """逐日截面 MAD 去极值。

    输入:
        df: 含 trade_date 的 panel
        cols: 因子列名列表
        n_mad: MAD 倍数

    输出:
        同形 DataFrame, 列被截断到 [median - n*MAD, median + n*MAD]
    """
    def _winz(g):
        med = g[cols].median()
        mad = (g[cols] - med).abs().median() * 1.4826
        lo, hi = med - n_mad * mad, med + n_mad * mad
        g[cols] = g[cols].clip(lower=lo, upper=hi, axis=1)
        return g
    return df.groupby("trade_date", group_keys=False).apply(_winz)


def cross_section_demean(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """逐日截面 demean。

    输入:
        df: 含 trade_date 的 panel
        cols: 因子列名列表

    输出:
        同形 DataFrame, 每列减去当日截面均值
    """
    def _dm(g):
        g[cols] = g[cols] - g[cols].mean()
        return g
    return df.groupby("trade_date", group_keys=False).apply(_dm)


def cross_section_zscore(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """逐日截面 z-score 标准化。

    输入:
        df: 含 trade_date 的 panel
        cols: 因子列名列表

    输出:
        同形 DataFrame, 每列 (x - mean) / std
    """
    def _zs(g):
        g[cols] = (g[cols] - g[cols].mean()) / g[cols].std(ddof=0).replace(0, 1)
        return g
    return df.groupby("trade_date", group_keys=False).apply(_zs)


print("[OK] 预处理函数定义完成")

In [ ]:
# ===== IC 筛选 & 共线去重（与 baseline 完全相同）=====

def compute_ic_panel(df: pd.DataFrame, feat_cols: List[str], label_col: str) -> pd.DataFrame:
    """逐日截面 Spearman IC 面板。

    输入:
        df: 含 trade_date 的 panel
        feat_cols: 特征列名列表
        label_col: 标签列名

    输出:
        DataFrame index=trade_date, columns=feat_cols, 值为日度 Spearman IC
    """
    out = {}
    for d, g in df.groupby("trade_date"):
        out[d] = g[feat_cols].corrwith(g[label_col], method="spearman")
    return pd.DataFrame(out).T.sort_index()


def summarize_ic(ic_panel: pd.DataFrame) -> pd.DataFrame:
    """IC 面板汇总统计。

    输入:
        ic_panel: compute_ic_panel 输出, index=date, columns=feat

    输出:
        DataFrame index=feat, columns=[ic_mean, ic_std, t_stat, pos_ratio]
    """
    mu = ic_panel.mean()
    sd = ic_panel.std()
    n = ic_panel.notna().sum()
    t = mu / sd * np.sqrt(n)
    pos = (ic_panel > 0).mean()
    return pd.DataFrame({"ic_mean": mu, "ic_std": sd, "t_stat": t, "pos_ratio": pos})


def filter_by_ic(ic_summary: pd.DataFrame, ic_th: float = 0.02, t_th: float = 2.0) -> List[str]:
    """按 |IC| 和 |t| 阈值过滤因子。

    输入:
        ic_summary: summarize_ic 输出
        ic_th: |IC mean| 下限
        t_th: |t_stat| 下限

    输出:
        通过筛选的因子名列表
    """
    mask = (ic_summary["ic_mean"].abs() > ic_th) & (ic_summary["t_stat"].abs() > t_th)
    return ic_summary.index[mask].tolist()


def spearman_union_find(df: pd.DataFrame, cols: List[str], corr_th: float = 0.7) -> List[List[str]]:
    """Spearman 相关矩阵 + 并查集分组, 将 |ρ| > corr_th 的因子聚为一组。

    输入:
        df: train DataFrame
        cols: 候选因子列名
        corr_th: 相关系数阈值

    输出:
        list[list[str]], 每个子表是一组共线因子
    """
    corr = df[cols].corr(method="spearman").abs().values
    np.fill_diagonal(corr, 0)
    n = len(cols)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    rows, ccols = np.where(corr > corr_th)
    for a, b in zip(rows, ccols):
        if a < b:
            union(a, b)

    groups = {}
    for i, c in enumerate(cols):
        groups.setdefault(find(i), []).append(c)
    return list(groups.values())


def select_representatives(groups: List[List[str]], ic_summary: pd.DataFrame) -> List[str]:
    """每组共线因子中选 |IC mean| 最大的作为代表。

    输入:
        groups: spearman_union_find 输出
        ic_summary: summarize_ic 输出

    输出:
        代表因子名列表
    """
    reps = []
    for grp in groups:
        if len(grp) == 1:
            reps.append(grp[0])
        else:
            best = ic_summary.loc[grp, "ic_mean"].abs().idxmax()
            reps.append(best)
    return reps


print("[OK] IC 筛选 & 共线去重函数定义完成")

In [ ]:
# ===== 建模核心：daily_regularized_beta（新增，替换 daily_ols_beta）=====

def daily_regularized_beta(
    df: pd.DataFrame,
    x_cols: List[str],
    y_col: str,
    model_type: str = "lasso",
    alpha: float = 0.001
) -> pd.DataFrame:
    """逐日截面 Lasso / Ridge 拟合，返回 beta 时间序列。

    输入:
        df: 含 trade_date 的 panel 数据
        x_cols: 特征列名列表
        y_col: 连续收益标签列名，例如 FWD_RET_10D
        model_type: "lasso" 或 "ridge"
        alpha: 正则化强度

    输出:
        DataFrame:
            index = trade_date
            columns = ["__intercept__"] + x_cols
            每一行是对应交易日横截面模型估计得到的截距和因子系数
    """
    assert model_type in ("lasso", "ridge"), f"model_type 必须是 lasso 或 ridge，当前: {model_type}"

    ModelClass = Lasso if model_type == "lasso" else Ridge

    rows = []
    for d, g in df.groupby("trade_date"):
        X = g[x_cols].values
        y = g[y_col].values

        # 过滤含 NaN / Inf 的样本
        mask = np.isfinite(X).all(axis=1) & np.isfinite(y)

        # 样本数太少时跳过，避免单日截面估计不稳定
        if mask.sum() < len(x_cols) + 5:
            continue

        model = ModelClass(alpha=alpha, fit_intercept=True, max_iter=5000)
        model.fit(X[mask], y[mask])

        beta = np.r_[model.intercept_, model.coef_.copy()]
        rows.append((d, beta))

    if not rows:
        return pd.DataFrame(columns=["__intercept__"] + x_cols)

    idx = [r[0] for r in rows]
    arr = np.array([r[1] for r in rows])
    return pd.DataFrame(arr, index=idx, columns=["__intercept__"] + x_cols)


print("[OK] daily_regularized_beta 定义完成")


In [ ]:
# ===== EWM 平滑 / Walk-Forward 预测 / IC 评估 =====

def smooth_beta_ewm(beta_df: pd.DataFrame, halflife: int = 20) -> pd.DataFrame:
    """beta 时间序列 EWM 平滑。

    输入:
        beta_df: daily_regularized_beta 输出, index=trade_date
        halflife: EWM 半衰期（交易日）

    输出:
        与 beta_df 同形的 DataFrame
    """
    return beta_df.ewm(halflife=halflife, adjust=False).mean()


def predict_walkforward(
    test_df: pd.DataFrame,
    x_cols: List[str],
    beta_ewm_shifted: pd.DataFrame,
    label_col: str
) -> pd.DataFrame:
    """测试日 t 使用 t-1 及以前可获得的平滑 beta 预测。

    输入:
        test_df: 测试集 panel 数据
        x_cols: 特征列名列表
        beta_ewm_shifted: EWM 平滑后再 shift(1) 的 beta 表
        label_col: 真实连续收益标签列名

    输出:
        DataFrame columns=[trade_date, ts_code, y_pred, y_true]
    """
    records = []

    required_beta_cols = ["__intercept__"] + x_cols

    for d, g in test_df.groupby("trade_date"):
        if d not in beta_ewm_shifted.index:
            continue

        beta_row = beta_ewm_shifted.loc[d]
        if not set(required_beta_cols).issubset(beta_row.index):
            continue

        intercept = beta_row["__intercept__"]
        b = beta_row[x_cols].values

        if not np.isfinite(intercept) or not np.isfinite(b).all():
            continue

        X = g[x_cols].values
        y_pred = np.full(len(g), np.nan)
        mask = np.isfinite(X).all(axis=1)
        y_pred[mask] = intercept + X[mask] @ b

        for i, (_, row) in enumerate(g.iterrows()):
            records.append({
                "trade_date": d,
                "ts_code": row["ts_code"],
                "y_pred": y_pred[i],
                "y_true": row[label_col]
            })

    return pd.DataFrame(records)


def make_rank_output(pred_df: pd.DataFrame) -> pd.DataFrame:
    """根据 y_pred 生成每日股票排序结果。

    输入:
        pred_df: predict_walkforward 输出

    输出:
        DataFrame columns=[trade_date, rank, stockid, reg_score]
        其中 reg_score 为模型预测得分，按每日从高到低排序。
    """
    if pred_df.empty:
        return pd.DataFrame(columns=["trade_date", "rank", "stockid", "reg_score"])

    out = pred_df.dropna(subset=["y_pred"]).copy()
    out["rank"] = out.groupby("trade_date")["y_pred"].rank(method="first", ascending=False).astype(int)
    out = out.rename(columns={"ts_code": "stockid", "y_pred": "reg_score"})
    return out[["trade_date", "rank", "stockid", "reg_score"]].sort_values(["trade_date", "rank"])


def evaluate_ic(pred_df: pd.DataFrame) -> Tuple[pd.Series, dict]:
    """日度 IC + 汇总评估。

    输入:
        pred_df: predict_walkforward 输出

    输出:
        (ic_series, summary_dict)
        ic_series: index=trade_date, 值=日度 Spearman IC
        summary_dict: ic_mean / ic_std / ic_ir / pos_ratio / n_days
    """
    if pred_df.empty or not {"y_pred", "y_true"}.issubset(pred_df.columns):
        return pd.Series(dtype=float), {}

    ic_list = []
    for d, g in pred_df.dropna(subset=["y_pred", "y_true"]).groupby("trade_date"):
        if len(g) < 5:
            continue
        ic, _ = spearmanr(g["y_pred"], g["y_true"])
        ic_list.append((d, ic))

    if not ic_list:
        return pd.Series(dtype=float), {}

    ic_series = pd.Series(dict(ic_list)).sort_index()
    n = len(ic_series)
    mu, sd = ic_series.mean(), ic_series.std()

    # 10 日收益标签下，一年约 252/10 个调仓周期
    ic_ir = (mu / sd) * np.sqrt(252 / 10) if sd > 0 else np.nan

    summary = {
        "ic_mean": mu,
        "ic_std": sd,
        "ic_ir": ic_ir,
        "pos_ratio": (ic_series > 0).mean(),
        "n_days": n
    }
    return ic_series, summary


print("[OK] Step 1 全部函数定义完成")


## Step 2 — 数据加载

读取 fold CSV，检查行列数。与 baseline 完全相同。

In [ ]:
train_df, test_df = load_fold(FOLD_DIR, FOLD_ID)
feature_cols, label_col = identify_columns(train_df)

print(f"训练集: {train_df.shape[0]:,} 行, {train_df['trade_date'].nunique()} 日")
print(f"  日期范围: {train_df['trade_date'].min().date()} ~ {train_df['trade_date'].max().date()}")
print(f"测试集:  {test_df.shape[0]:,} 行, {test_df['trade_date'].nunique()} 日")
print(f"  日期范围: {test_df['trade_date'].min().date()} ~ {test_df['trade_date'].max().date()}")
print(f"因子列数: {len(feature_cols)}")
print(f"标签列: {label_col}")
print(f"训练集标签缺失率: {train_df[label_col].isna().mean():.2%}")
print(f"测试集标签缺失率:  {test_df[label_col].isna().mean():.2%}")

## Step 3 — 截面预处理

train 和 test **各自独立**处理，防止数据泄漏。三步流水线：MAD Winsorize → Demean → Z-score。

与 baseline 完全相同，包括缺失值策略（不填充，NaN 自然传播）。

In [ ]:
print("3a. MAD Winsorize...")
train_df = cross_section_winsorize(train_df, feature_cols, WINSORIZE_MAD)
test_df  = cross_section_winsorize(test_df,  feature_cols, WINSORIZE_MAD)

print("3b. Demean...")
train_df = cross_section_demean(train_df, feature_cols)
test_df  = cross_section_demean(test_df,  feature_cols)

print("3c. Z-score...")
train_df = cross_section_zscore(train_df, feature_cols)
test_df  = cross_section_zscore(test_df,  feature_cols)

print("[OK] 截面预处理完成")

## Step 4 — IC 初筛

**只用训练集**计算 Spearman IC，过滤 `|ic_mean| > 0.02` 且 `|t_stat| > 2.0` 的因子。

与 baseline 完全相同（IC 筛选与模型类型无关）。

In [ ]:
print("计算训练集 IC 面板（可能需要几分钟）...")
ic_panel  = compute_ic_panel(train_df, feature_cols, label_col)
ic_summary = summarize_ic(ic_panel)
kept_ic   = filter_by_ic(ic_summary, IC_THRESHOLD, T_STAT_THRESHOLD)

print(f"[OK] IC 初筛: {len(feature_cols)} → {len(kept_ic)} 个因子")
print(f"  IC 均值范围: [{ic_summary.loc[kept_ic,'ic_mean'].min():.4f}, {ic_summary.loc[kept_ic,'ic_mean'].max():.4f}]")
print(f"  t 统计量范围: [{ic_summary.loc[kept_ic,'t_stat'].abs().min():.2f}, {ic_summary.loc[kept_ic,'t_stat'].abs().max():.2f}]")

## Step 5 — 共线性去重

Spearman 相关矩阵 + 并查集，`|ρ| > 0.7` 的因子聚为一组，每组保留 IC 最大的代表。

与 baseline 完全相同。

In [ ]:
print("共线性分组...")
groups     = spearman_union_find(train_df, kept_ic, CORR_THRESHOLD)
kept_final = select_representatives(groups, ic_summary)

print(f"[OK] 共线去重: {len(kept_ic)} → {len(kept_final)} 个因子")
group_sizes = sorted([len(g) for g in groups], reverse=True)
print(f"  最大共线组大小: {group_sizes[0]}")
print(f"  保留因子（前10）: {kept_final[:10]}")

## Step 6 — 逐日正则化回归

**核心改动**：用 `daily_regularized_beta` 替换 baseline 的 `daily_ols_beta`。

- `MODEL_TYPE = "lasso"` 时：L1 惩罚，部分系数被压缩至 0
- `MODEL_TYPE = "ridge"` 时：L2 惩罚，所有系数变小但保持非零

输出格式与 baseline 完全一致，后续步骤无需改动。

In [ ]:
x_cols = kept_final

print(f"训练集逐日 {MODEL_TYPE.upper()} 拟合（alpha={ALPHA}）...")
beta_train = daily_regularized_beta(train_df, x_cols, label_col, MODEL_TYPE, ALPHA)
print(f"  训练集 beta: {beta_train.shape}")

# 防泄漏处理：
# 不使用 test_df 的真实收益标签拟合 beta_test。
# 测试期预测只使用训练期最后可获得的 beta，经 EWM 平滑后向测试日期前向填充。
if MODEL_TYPE == "lasso" and len(beta_train) > 0:
    coef_cols = [c for c in beta_train.columns if c != "__intercept__"]
    nonzero_ratio = (beta_train[coef_cols].abs() > 1e-8).mean().mean()
    print(f"  Lasso 非零系数比例（训练集均值）: {nonzero_ratio:.2%}")

print(f"[OK] Step 6 完成")


## Step 7 — EWM 平滑 + Shift（与 baseline 完全相同）

`shift(1)` 是防泄漏核心：确保 t 日预测只用 t-1 日及之前的信息。

In [ ]:
beta_ewm = smooth_beta_ewm(beta_train, EWM_HALFLIFE)

# 将测试日期纳入 beta 表索引，并用训练期最后的 beta 向前填充。
# 再 shift(1)，保证任一测试日 t 只能使用 t-1 及以前的信息。
test_dates = sorted(test_df["trade_date"].unique())
combined_idx = sorted(set(beta_ewm.index.tolist()) | set(test_dates))
beta_extended = beta_ewm.reindex(combined_idx).ffill()
beta_for_pred = beta_extended.shift(1)

# 为了命名兼容，beta_all 表示“可用于保存的历史 beta”，这里只包含训练期拟合结果。
beta_all = beta_train.copy()

print(f"训练期 beta: {beta_train.shape}")
print(f"EWM 平滑后: {beta_ewm.shape}")
print(f"扩展到测试日期后: {beta_extended.shape}")
print(f"[OK] Step 7 完成（测试期未使用真实标签拟合 beta）")


## Step 8 — Walk-Forward 预测（与 baseline 完全相同）

对测试集每天，用 `β_ewm[t-1]` 对当天所有股票打分。

In [ ]:
pred_df = predict_walkforward(test_df, x_cols, beta_for_pred, label_col)

print(f"预测结果: {len(pred_df):,} 行")
if len(pred_df) > 0:
    print(f"  y_pred 范围: [{pred_df['y_pred'].min():.4f}, {pred_df['y_pred'].max():.4f}]")
    print(f"  y_pred NaN 数: {pred_df['y_pred'].isna().sum()}")
print(f"[OK] Step 8 完成")

## Step 9 — IC 评估（与 baseline 完全相同）

日度 Spearman IC + 汇总指标，与 baseline 结果对比。

In [ ]:
if len(pred_df) > 0:
    ic_series, ic_summary_dict = evaluate_ic(pred_df)

    print(f"=== IC 汇总 [{MODEL_TYPE.upper()}, alpha={ALPHA}] ===")
    for k, v in ic_summary_dict.items():
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

    print(f"\n=== 与 Baseline (OLS) 对比 ===")
    baseline_ref = {"ic_mean": 0.1357, "ic_ir": 5.66, "pos_ratio": 0.8796}
    for k, bv in baseline_ref.items():
        mv = ic_summary_dict.get(k, float('nan'))
        diff = mv - bv
        sign = "+" if diff >= 0 else ""
        print(f"  {k}: {MODEL_TYPE.upper()}={mv:.4f}  OLS={bv:.4f}  差异={sign}{diff:.4f}")

    # IC 序列图
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(range(len(ic_series)), ic_series.values,
               color=['#d62728' if v < 0 else '#2ca02c' for v in ic_series.values], alpha=0.7)
        ax.axhline(y=0, color='black', linewidth=0.5)
        ax.axhline(y=ic_series.mean(), color='#1f77b4', linestyle='--',
                   label=f"Mean IC={ic_series.mean():.4f}")
        ax.axhline(y=0.1357, color='gray', linestyle=':', alpha=0.6,
                   label="Baseline OLS Mean IC=0.1357")
        ax.set_title(f"Fold {FOLD_ID} — Daily Spearman IC ({MODEL_TYPE.upper()}, alpha={ALPHA})")
        ax.set_xlabel("Test day index")
        ax.set_ylabel("Spearman IC")
        ax.legend()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"  [图表跳过] {e}")
else:
    print("[ERROR] 无预测结果，跳过 IC 评估")
    ic_series = pd.Series(dtype=float)
    ic_summary_dict = {}

## Step 10 — 持久化输出

输出目录：`02_线性模型lasso_ridge/output/{date}/`，文件格式与 baseline 完全一致。

summary.csv 额外记录 `model_type` 和 `alpha`，方便后续多次实验对比。

In [ ]:
run_date = _dt.date.today().strftime("%Y%m%d")
out_dir = OUTPUT_DIR / run_date / f"{MODEL_TYPE}_alpha{ALPHA}"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"输出目录: {out_dir}")

if len(pred_df) > 0:
    # 1) y_pred
    pred_path = out_dir / f"fold_{FOLD_ID}_y_pred.parquet"
    pred_df.to_parquet(pred_path, index=False)
    print(f"[SAVED] {pred_path}  ({pred_df.shape[0]:,} rows)")

    # 2) rank output：满足任务清单中的 rank / stockid / reg_score 格式
    rank_df = make_rank_output(pred_df)
    rank_path = out_dir / f"fold_{FOLD_ID}_rank_output.csv"
    rank_df.to_csv(rank_path, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {rank_path}  ({rank_df.shape[0]:,} rows)")

    # 3) IC series
    ic_path = out_dir / f"fold_{FOLD_ID}_ic_series.csv"
    ic_series.to_csv(ic_path, header=["ic"])
    print(f"[SAVED] {ic_path}")

    # 4) Summary
    summary_row = {
        "model_type": MODEL_TYPE,
        "alpha": ALPHA,
        "ic_mean": ic_summary_dict.get("ic_mean", np.nan),
        "ic_std": ic_summary_dict.get("ic_std", np.nan),
        "ic_ir": ic_summary_dict.get("ic_ir", np.nan),
        "pos_ratio": ic_summary_dict.get("pos_ratio", np.nan),
        "n_days": ic_summary_dict.get("n_days", 0),
        "n_features_kept": len(kept_final),
        "tuning_mode": "fixed_alpha" if "ALPHA_GRID" not in globals() else "train_validation_alpha_search"
    }
    summary_path = out_dir / f"fold_{FOLD_ID}_summary.csv"
    pd.DataFrame([summary_row]).to_csv(summary_path, index=False)
    print(f"[SAVED] {summary_path}")

    # 5) Kept factors
    factors_path = out_dir / f"fold_{FOLD_ID}_kept_factors.txt"
    factors_path.write_text("\n".join(kept_final), encoding="utf-8")
    print(f"[SAVED] {factors_path}  ({len(kept_final)} factors)")

    # 6) Beta history（只包含训练期拟合 beta；测试期不拟合 beta）
    beta_path = out_dir / f"fold_{FOLD_ID}_beta_history.parquet"
    beta_all.to_parquet(beta_path)
    print(f"[SAVED] {beta_path}  ({beta_all.shape[0]} days x {beta_all.shape[1]} cols)")

    print(f"\n[DONE] Fold {FOLD_ID} [{MODEL_TYPE.upper()}, alpha={ALPHA}] 全部输出完成 -> {out_dir}")
else:
    print("[ERROR] 无有效预测结果，跳过输出")
